# Stage B2 — Baseline Training

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.2.1 — the no-physics baseline that Phase D
compares the PINN against.

**No physics here either** (same note as B1): plain MLP, MSE loss,
early stopping. This is a single, final fit of the architecture B1
selected — not a search — so it gets a longer patience (200 epochs)
than B1's CV search used (30), since there's only one fit to get right
here instead of 240.

Self-contained for data (reloads/re-splits `masters_data.xlsx` exactly
as A3/B1 do), but **not** for the architecture: this notebook reads
`outputs/B1_selected_architecture.json`, so run B1's save cell first.

**Input:** `data/masters_data.xlsx`, `outputs/B1_selected_architecture.json`
**Output:** trained model, training-curve and prediction figures,
`outputs/B2_predictions.csv` (train/val/test predictions in both
normalized and physical units) and the saved model weights — this is
what Phase D loads to compare against the PINN.


## Setup

In [ ]:
import json
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH


## 1. Load, split, normalize (same logic as A3/B1)

Repeated here rather than imported so this notebook runs standalone
for the data side.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]

medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)

extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2

rng = np.random.default_rng(SEED)
jitter = rng.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))

train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])

def split_arrays(split_name, cols):
    sub = df.filter(pl.col("split") == split_name)
    return sub.select([f"{c}_norm" for c in cols]).to_numpy()

X_train, Y_train = split_arrays("train", INPUT_COLS), split_arrays("train", OUTPUT_COLS)
X_val, Y_val = split_arrays("val", INPUT_COLS), split_arrays("val", OUTPUT_COLS)
X_test, Y_test = split_arrays("test", INPUT_COLS), split_arrays("test", OUTPUT_COLS)
print(f"train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")


## 2. Load the architecture B1 selected

Falls back to the linear baseline (`A13_linear`, no hidden layer) if
B1 hasn't been run yet — B1's own shadow-verified run selected exactly
that, so it's a documented placeholder, not an arbitrary guess, but
**run B1 for the real TensorFlow-based selection before trusting this
number for the dissertation.**

In [ ]:
SELECTION_PATH = OUT_DIR / "B1_selected_architecture.json"

if SELECTION_PATH.exists():
    with open(SELECTION_PATH) as f:
        selection = json.load(f)
    print(f"Loaded B1 selection: {selection}")
else:
    selection = {"architecture_id": "A13_linear (fallback default)", "hidden_units": [], "n_params": None}
    print("WARNING: B1_selected_architecture.json not found -- using a fallback default.")
    print("Run B1's save cell first for the real selection. Using:", selection)

HIDDEN_UNITS = tuple(selection["hidden_units"])
print(f"\nArchitecture: {selection['architecture_id']}  hidden_units={HIDDEN_UNITS}")


## 3. Build & compile

Same builder as B1, same equation:

$$
h^{(l)} = \tanh\!\left( W^{(l)} h^{(l-1)} + b^{(l)} \right), \qquad
\hat{y} = W^{(L+1)} h^{(L)} + b^{(L+1)}
$$

If `HIDDEN_UNITS` is empty (the linear case), the loop below adds no
`tanh` layers and the model reduces to $\hat y = Wx + b$ — no special
case needed, it falls out of the same code.

In [ ]:
def build_model(hidden_units, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    model = keras.Sequential([keras.Input(shape=(input_dim,))])
    for units in hidden_units:
        model.add(layers.Dense(units, activation="tanh"))
    model.add(layers.Dense(output_dim, activation="linear"))
    model.compile(optimizer="adam", loss="mse")
    return model

model = build_model(HIDDEN_UNITS, seed=SEED)
model.summary()


## 4. Train

Patience of 200 (vs. B1's 30) since this is the one fit that has to be
right, not 240 quick comparisons.

In [ ]:
EPOCHS = 3000
PATIENCE = 200
BATCH_SIZE = 8

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=PATIENCE, restore_best_weights=True
)
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[early_stop],
)
n_epochs_run = len(history.history["loss"])
print(f"Stopped after {n_epochs_run} epochs "
      f"({'early stopping triggered' if n_epochs_run < EPOCHS else 'hit EPOCHS cap -- consider raising it'})")


## 5. Training curves

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history["loss"], mode="lines", name="train loss",
                          line=dict(color="#185FA5", width=2)))
fig.add_trace(go.Scatter(y=history.history["val_loss"], mode="lines", name="val loss",
                          line=dict(color="#993C1D", width=2)))
best_epoch = int(np.argmin(history.history["val_loss"]))
fig.add_vline(x=best_epoch, line_dash="dot", line_color="#5A5A55",
              annotation_text=f"best val_loss @ epoch {best_epoch}")
fig.update_layout(title="B2 baseline training curves", xaxis_title="epoch",
                   yaxis_title="MSE (normalized outputs)", width=800, height=420)
fig.show()


## 6. Predictions on train / val / test

In [ ]:
pred_train = model.predict(X_train, verbose=0)
pred_val = model.predict(X_val, verbose=0)
pred_test = model.predict(X_test, verbose=0)
print("predicted shapes:", pred_train.shape, pred_val.shape, pred_test.shape)


## 7. Predicted vs. observed, per output

In [ ]:
fig = make_subplots(rows=2, cols=3, subplot_titles=OUTPUT_COLS)
split_colors = {"train": "#B7C9DA", "val": "#854F0B", "test": "#993C1D"}
preds = {"train": pred_train, "val": pred_val, "test": pred_test}
truths = {"train": Y_train, "val": Y_val, "test": Y_test}

for i, out in enumerate(OUTPUT_COLS):
    r, c = divmod(i, 3)
    for s, color in split_colors.items():
        fig.add_trace(
            go.Scatter(x=truths[s][:, i], y=preds[s][:, i], mode="markers",
                       marker=dict(color=color, size=8), name=s, showlegend=(i == 0)),
            row=r + 1, col=c + 1,
        )
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                              line=dict(color="#B0AFA8", dash="dot"), showlegend=False),
                  row=r + 1, col=c + 1)

fig.update_layout(height=650, width=950,
                   title_text="Predicted vs. observed (normalized), per output")
fig.show()


## 8. Quick sanity-check metrics

$$
R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}
$$

per output, per split — not the full Eq. 3.21–3.27 treatment (that's
Phase D's job, comparing this baseline against the PINN), just enough
to confirm training actually worked before moving on.

In [ ]:
rows = []
for s, (X, Y, pred) in {"train": (X_train, Y_train, pred_train),
                         "val": (X_val, Y_val, pred_val),
                         "test": (X_test, Y_test, pred_test)}.items():
    for i, out in enumerate(OUTPUT_COLS):
        ss_res = np.sum((Y[:, i] - pred[:, i]) ** 2)
        ss_tot = np.sum((Y[:, i] - Y[:, i].mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
        rows.append({"split": s, "output": out, "r2": r2})
metrics = pl.DataFrame(rows)

pivot_rows = []
for out in OUTPUT_COLS:
    row = {"output": out}
    for s in ["train", "val", "test"]:
        row[s] = metrics.filter((pl.col("output") == out) & (pl.col("split") == s))["r2"][0]
    pivot_rows.append(row)
pl.DataFrame(pivot_rows)


**Same table, as a picture:**

In [ ]:
fig = go.Figure()
for s, color in split_colors.items():
    sub = metrics.filter(pl.col("split") == s)
    fig.add_trace(go.Bar(x=sub["output"], y=sub["r2"], name=s, marker_color=color))
fig.add_hline(y=0, line_color="#5A5A55", line_width=1)
fig.update_layout(barmode="group", title="R\u00b2 per output and split (baseline, no physics)",
                   yaxis_title="R\u00b2", width=800, height=420)
fig.show()


## Optional — persist outputs

Saves predictions (normalized + back-transformed to physical units,
using the train-only min/max from Section 1) and the trained model —
both are what Phase D (D1) loads to compare against the PINN.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred_rows = []
for s, (idx_df, pred) in {
    "train": (df.filter(pl.col("split") == "train"), pred_train),
    "val": (df.filter(pl.col("split") == "val"), pred_val),
    "test": (df.filter(pl.col("split") == "test"), pred_test),
}.items():
    for j in range(pred.shape[0]):
        row = {"split": s}
        for i, out in enumerate(OUTPUT_COLS):
            norm_val = float(pred[j, i])
            phys_val = norm_val * (train_max[out] - train_min[out]) + train_min[out]
            row[f"{out}_pred_norm"] = norm_val
            row[f"{out}_pred"] = phys_val
        pred_rows.append(row)

pl.DataFrame(pred_rows).write_csv(OUT_DIR / "B2_predictions.csv")
try:
    model.save(OUT_DIR / "B2_baseline_model.keras")
except Exception as e:
    print(f"'.keras' save failed ({e}); falling back to '.h5'")
    model.save(OUT_DIR / "B2_baseline_model.h5")
metrics.write_csv(OUT_DIR / "B2_metrics.csv")
print(f"Saved to {OUT_DIR}")


## Next

**C1** starts Phase C on the same architecture shape (`HIDDEN_UNITS`
above), adding the physics constraints (Table 7) that this baseline
deliberately doesn't have. Phase D (D1) will load both this baseline's
predictions and the PINN ensemble's to run the actual comparison.
